# UPD evaluation for popularity calibration

User Popularity Deviation (UPD) measures the difference between users' historical popularity preferences and the popularity mix of their recommendation lists. This evaluation applies to unadjusted ALS recommendations, fixed popularity calibration, and history-aware calibration.

**Current implementation:** independent Spark initialization, validated popularity mixes, per-user Jensen–Shannon divergence using base-2 logarithms, validated segment and overall aggregation, synthetic correctness checks, and session shutdown. The evaluator accepts prepared input tables and validates them through a complete synthetic run. Real-data UPD results remain pending the required prepared inputs; all displayed scores are synthetic test results.

Run the code cells in order using the **CS 7265 - Scala 2.12.18** kernel with Java 17. These checks do not require MovieLens, recommendation exports, or an active baseline notebook session.

The metric follows [the source paper, Section 3.1.2, equations 4–5](https://arxiv.org/html/2103.06364#S3.SS1.SSS2). Base-2 logarithms are an explicit implementation convention, giving Jensen–Shannon divergence a theoretical range of 0 to 1. Overall UPD averages individual scores within user segments and then averages the three segment scores with equal segment weight.


## 1. Verify the local environment

The `spark-sql` dependency provides Spark sessions and DataFrames for evaluation. Spark 3.5.9 matches the baseline notebook. The application runs locally with two worker threads and binds the driver to the loopback address.

Counting three synthetic rows verifies that Spark can execute an operation. Assertions check the Spark, Scala, and Java versions and the resulting count. The output is an environment check, not a UPD result.


In [1]:
import $ivy.`org.apache.spark::spark-sql:3.5.9`

import org.apache.spark.sql.SparkSession


import $ivy.$
import org.apache.spark.sql.SparkSession

In [2]:
val spark = SparkSession.builder()
  .appName("CS 7265 - UPD Evaluation")
  .master("local[2]")
  .config("spark.driver.host", "127.0.0.1")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
val scalaVersion = scala.util.Properties.versionNumberString
val javaVersion = System.getProperty("java.version")
val checkCount = spark.range(3).count()

assert(spark.version == "3.5.9")
assert(scalaVersion == "2.12.18")
assert(System.getProperty("java.specification.version") == "17")
assert(checkCount == 3L)

println(s"Spark: ${spark.version}")
println(s"Scala: $scalaVersion")
println(s"Java: $javaVersion")
println(s"Environment check passed: $checkCount synthetic rows.")


Spark: 3.5.9


Scala: 2.12.18


Java: 17.0.16


Environment check passed: 3 synthetic rows.


spark: SparkSession = org.apache.spark.sql.SparkSession@1f057e72
scalaVersion: String = "2.12.18"
javaVersion: String = "17.0.16"
checkCount: Long = 3L

## 2. Define and validate popularity mixes

A `PopularityMix` contains the Head, Mid, and Tail proportions for one historical profile or recommendation list. Historical proportions are weighted by observed training-rating values; recommendation proportions are category counts divided by list size. This record accepts those prepared proportions rather than deriving them from source files.

Each proportion must be finite and between 0 and 1. The total must be within `1e-9` of 1 to tolerate floating-point rounding. Zero shares are valid. Invalid records raise `IllegalArgumentException`; inputs are not silently normalized.


In [3]:
case class PopularityMix(
  head: Double,
  mid: Double,
  tail: Double
) {
  private val proportions = Seq(head, mid, tail)

  require(
    proportions.forall(value =>
      !value.isNaN &&
      !value.isInfinity &&
      value >= 0.0 &&
      value <= 1.0
    ),
    "Popularity proportions must be finite numbers between 0 and 1."
  )

  require(
    math.abs(proportions.sum - 1.0) <= 1e-9,
    "Popularity proportions must sum to 1."
  )
}


defined class PopularityMix

## 3. Calculate individual popularity mismatch

The midpoint is the category-by-category average of the historical and recommendation mixes. It serves as a common reference for both comparisons; it does not modify either input or define a new recommendation target.


In [4]:
def midpoint(
  historical: PopularityMix,
  recommended: PopularityMix
): PopularityMix = {
  PopularityMix(
    head = (historical.head + recommended.head) / 2.0,
    mid  = (historical.mid  + recommended.mid)  / 2.0,
    tail = (historical.tail + recommended.tail) / 2.0
  )
}


defined function midpoint

For each category, the KL contribution is the input proportion multiplied by the base-2 logarithm of its ratio to the midpoint proportion. A zero input share contributes zero without evaluating `log(0)`. For a positive share, its midpoint share is also positive.

`categoryContribution` and `klAgainstMidpoint` are helpers for comparisons with the constructed midpoint. Individual category contributions retain their signs. `popularityMismatch` averages the two KL totals to produce one user's Jensen–Shannon divergence. It returns divergence.


In [5]:
def categoryContribution(
  proportion: Double,
  reference: Double
): Double = {
  if (proportion == 0.0) {
    0.0
  } else {
    proportion * (math.log(proportion / reference) / math.log(2.0))
  }
}

def klAgainstMidpoint(
  mix: PopularityMix,
  reference: PopularityMix
): Double = {
  categoryContribution(mix.head, reference.head) +
  categoryContribution(mix.mid,  reference.mid) +
  categoryContribution(mix.tail, reference.tail)
}

def popularityMismatch(
  historical: PopularityMix,
  recommended: PopularityMix
): Double = {
  val reference = midpoint(historical, recommended)

  val historicalDifference =
    klAgainstMidpoint(historical, reference)

  val recommendationDifference =
    klAgainstMidpoint(recommended, reference)

  (historicalDifference + recommendationDifference) / 2.0
}


defined function categoryContribution
defined function klAgainstMidpoint
defined function popularityMismatch

## 4. Verify with synthetic examples

These invented mixes test the calculation independently of teammate inputs. Identical mixes must score zero; disjoint all-Head and all-Tail mixes must score one with base-2 logarithms. Swapping inputs must preserve the score. A category absent from both mixes must remain safe to evaluate.

The worked example compares `[0.2, 0.3, 0.5]` with `[0.7, 0.2, 0.1]`. Its expected divergence is approximately `0.2183635969629476`. Assertions use a `1e-9` tolerance. Additional checks verify rejection of invalid proportions and acceptance of a negligible rounding difference in the total.


In [6]:
val tolerance = 1e-9
val historicalMix = PopularityMix(0.2, 0.3, 0.5)
val recommendationMix = PopularityMix(0.7, 0.2, 0.1)

val matchingScore = popularityMismatch(historicalMix, historicalMix)
assert(math.abs(matchingScore) <= tolerance,
  "Identical mixes should have zero mismatch.")

val allHead = PopularityMix(1.0, 0.0, 0.0)
val allTail = PopularityMix(0.0, 0.0, 1.0)
val oppositeScore = popularityMismatch(allHead, allTail)
assert(math.abs(oppositeScore - 1.0) <= tolerance,
  "Non-overlapping mixes should have mismatch 1 with base-2 logs.")
assert(math.abs(popularityMismatch(allHead, allHead)) <= tolerance,
  "Identical mixes with shared zero categories should score zero.")

val forwardScore = popularityMismatch(historicalMix, recommendationMix)
val reverseScore = popularityMismatch(recommendationMix, historicalMix)
assert(math.abs(forwardScore - reverseScore) <= tolerance,
  "Swapping the mixes should not change their mismatch.")
assert(math.abs(forwardScore - 0.2183635969629476) <= tolerance,
  "The worked example should match its independently calculated score.")
assert(!forwardScore.isNaN && !forwardScore.isInfinity &&
  forwardScore >= 0.0 && forwardScore <= 1.0,
  "The example score should be finite and within 0 to 1.")

val invalidInputs = Seq(
  (0.2, 0.3, 0.4),
  (-0.1, 0.6, 0.5),
  (1.1, 0.0, 0.0),
  (Double.NaN, 0.5, 0.5),
  (Double.PositiveInfinity, 0.0, 0.0),
  (Double.NegativeInfinity, 0.0, 0.0)
)
invalidInputs.foreach { case (head, mid, tail) =>
  val rejected = try {
    PopularityMix(head, mid, tail)
    false
  } catch {
    case _: IllegalArgumentException => true
  }
  assert(rejected, s"Invalid mix should be rejected: ($head, $mid, $tail)")
}
val roundingExample = PopularityMix(0.2, 0.3, 0.5000000000000001)

println(f"Matching mixes: $matchingScore%.10f")
println(f"Non-overlapping mixes: $oppositeScore%.10f")
println(f"Worked example: $forwardScore%.10f")
println(f"Reversed example: $reverseScore%.10f")
println("All synthetic mismatch and input-validation checks passed.")


Matching mixes: 0.0000000000


Non-overlapping mixes: 1.0000000000


Worked example: 0.2183635970


Reversed example: 0.2183635970


All synthetic mismatch and input-validation checks passed.


tolerance: Double = 1.0E-9
historicalMix: PopularityMix = PopularityMix(0.2, 0.3, 0.5)
recommendationMix: PopularityMix = PopularityMix(0.7, 0.2, 0.1)
matchingScore: Double = 0.0
allHead: PopularityMix = PopularityMix(1.0, 0.0, 0.0)
allTail: PopularityMix = PopularityMix(0.0, 0.0, 1.0)
oppositeScore: Double = 1.0
forwardScore: Double = 0.2183635969629475
reverseScore: Double = 0.2183635969629475
invalidInputs: Seq[(Double, Double, Double)] = List(
  (0.2, 0.3, 0.4),
  (-0.1, 0.6, 0.5),
  (1.1, 0.0, 0.0),
  (NaN, 0.5, 0.5),
  (Infinity, 0.0, 0.0),
  (-Infinity, 0.0, 0.0)
)
roundingExample: PopularityMix = PopularityMix(0.2, 0.3, 0.5000000000000001)

## 5. Validate scores and aggregate UPD

Aggregation consumes one row per evaluation user for one recommendation method and one experiment run. The input columns are `userId` (integer), `segment` (string), and `mismatch` (double). Segment labels are `blockbuster-focused`, `diverse`, and `niche-focused`.

The validation rejects missing fields, non-finite or out-of-range scores, repeated users, unexpected labels, and missing segments. This prevents null scores from silently disappearing from Spark's average and duplicate users from receiving extra weight. The reusable validation function also allows malformed synthetic inputs to exercise the same checks. Only distinct segment labels are collected locally.

These checks cannot establish complete evaluation-user coverage without a separate expected-user list. Real-data integration must check that list before aggregation; it is not supplied by these synthetic examples.


In [7]:
import spark.implicits._
import org.apache.spark.sql.DataFrame
import org.apache.spark.sql.functions.{avg, col, count, isnan, lit}

def validateUserMismatchScores(scores: DataFrame): Unit = {
  val invalidRows = scores.filter(
    col("userId").isNull ||
    col("segment").isNull ||
    col("mismatch").isNull ||
    isnan(col("mismatch")) ||
    col("mismatch") < 0.0 ||
    col("mismatch") > 1.0
  )
  assert(invalidRows.count() == 0L,
    "Every row must have a user, a segment, and a finite mismatch score between 0 and 1.")

  val duplicateUsers = scores.groupBy("userId").count()
    .filter(col("count") > 1L)
  assert(duplicateUsers.count() == 0L,
    "Each user must contribute exactly one mismatch score.")

  val expectedSegments = Set("blockbuster-focused", "diverse", "niche-focused")
  val actualSegments = scores.select("segment").distinct()
    .as[String].collect().toSet
  assert(actualSegments == expectedSegments,
    "Evaluation must contain exactly the three expected user segments.")
}


import spark.implicits._
import org.apache.spark.sql.DataFrame
import org.apache.spark.sql.functions.{avg, col, count, isnan, lit}
defined function validateUserMismatchScores

### Aggregate a known example

The six scores below are invented, not outputs from the ALS baseline. Supplying known scores isolates the aggregation from the individual mismatch calculation. Expected segment means are 0.15, 0.25, and 0.45; expected overall UPD is 17/60, approximately 0.2833333333.

`groupBy` creates the segment groups. `count` records their sizes, and `avg` computes each segment's mean mismatch. A second `avg` gives each segment equal weight in overall UPD. All assertions run before interpreting the displayed results.


In [8]:
val userMismatchScores = Seq(
  (1, "blockbuster-focused", 0.10),
  (2, "blockbuster-focused", 0.20),
  (3, "diverse",             0.20),
  (4, "diverse",             0.30),
  (5, "niche-focused",       0.40),
  (6, "niche-focused",       0.50)
).toDF("userId", "segment", "mismatch")

validateUserMismatchScores(userMismatchScores)

val segmentUPD = userMismatchScores
  .groupBy("segment")
  .agg(count("*").as("userCount"), avg("mismatch").as("upd"))

val overallUPD = segmentUPD.agg(avg("upd").as("overallUPD"))

val expectedSegmentMeans = Map(
  "blockbuster-focused" -> 0.15,
  "diverse" -> 0.25,
  "niche-focused" -> 0.45
)
val segmentResults = segmentUPD.collect()
assert(segmentResults.length == 3)
segmentResults.foreach { row =>
  assert(row.getAs[Long]("userCount") == 2L)
  assert(math.abs(row.getAs[Double]("upd") -
    expectedSegmentMeans(row.getAs[String]("segment"))) <= tolerance)
}
assert(math.abs(overallUPD.first().getAs[Double]("overallUPD") - 17.0 / 60.0) <= tolerance)

segmentUPD.orderBy("segment").show(truncate = false)
overallUPD.show(truncate = false)
println("Known segment and overall UPD checks passed.")


+-------------------+---------+-------------------+
|segment            |userCount|upd                |
+-------------------+---------+-------------------+
|blockbuster-focused|2        |0.15000000000000002|
|diverse            |2        |0.25               |
|niche-focused      |2        |0.45               |
+-------------------+---------+-------------------+



+------------------+
|overallUPD        |
+------------------+
|0.2833333333333334|
+------------------+



Known segment and overall UPD checks passed.


userMismatchScores: DataFrame = [userId: int, segment: string ... 1 more field]
segmentUPD: DataFrame = [segment: string, userCount: bigint ... 1 more field]
overallUPD: DataFrame = [overallUPD: double]
expectedSegmentMeans: Map[String, Double] = Map(
  "blockbuster-focused" -> 0.15,
  "diverse" -> 0.25,
  "niche-focused" -> 0.45
)
segmentResults: Array[org.apache.spark.sql.Row] = Array(
  [diverse,2,0.25],
  [blockbuster-focused,2,0.15000000000000002],
  [niche-focused,2,0.45]
)

## 6. Verify aggregation safeguards

An unequal-size example distinguishes equal segment weighting from a pooled user average: one blockbuster-focused user scores 0, one diverse user scores 0, and four niche-focused users each score 1. The segment means are 0, 0, and 1, so overall UPD must be 1/3. Pooling all six users would give 2/3 and would not implement the specified aggregation.

Invalid-input examples verify that duplicate users, missing or unknown segments, missing fields, invalid scores, and empty inputs are rejected rather than silently averaged. The original valid example is preserved.


In [9]:
val unequalScores = Seq(
  (1, "blockbuster-focused", 0.0),
  (2, "diverse",             0.0),
  (3, "niche-focused",       1.0),
  (4, "niche-focused",       1.0),
  (5, "niche-focused",       1.0),
  (6, "niche-focused",       1.0)
).toDF("userId", "segment", "mismatch")

validateUserMismatchScores(unequalScores)
val unequalSegmentUPD = unequalScores.groupBy("segment")
  .agg(count("*").as("userCount"), avg("mismatch").as("upd"))
val unequalOverall = unequalSegmentUPD.agg(avg("upd")).first().getDouble(0)
val pooledUserAverage = unequalScores.agg(avg("mismatch")).first().getDouble(0)
assert(math.abs(unequalOverall - 1.0 / 3.0) <= tolerance)
assert(math.abs(pooledUserAverage - 2.0 / 3.0) <= tolerance)
assert(math.abs(unequalOverall - pooledUserAverage) > tolerance,
  "Overall UPD must weight segments equally, not pool users across segments.")

val invalidScoreTables = Seq(
  "duplicate user" -> userMismatchScores.union(userMismatchScores.filter(col("userId") === 1)),
  "missing segment" -> userMismatchScores.filter(col("segment") =!= "niche-focused"),
  "unknown segment" -> userMismatchScores.withColumn("segment", lit("unknown")),
  "missing user" -> userMismatchScores.withColumn("userId", lit(null).cast("int")),
  "missing label" -> userMismatchScores.withColumn("segment", lit(null).cast("string")),
  "missing score" -> userMismatchScores.withColumn("mismatch", lit(null).cast("double")),
  "NaN score" -> userMismatchScores.withColumn("mismatch", lit(Double.NaN)),
  "positive infinity" -> userMismatchScores.withColumn("mismatch", lit(Double.PositiveInfinity)),
  "negative infinity" -> userMismatchScores.withColumn("mismatch", lit(Double.NegativeInfinity)),
  "negative score" -> userMismatchScores.withColumn("mismatch", lit(-0.1)),
  "score above one" -> userMismatchScores.withColumn("mismatch", lit(1.1)),
  "empty input" -> userMismatchScores.limit(0)
)
invalidScoreTables.foreach { case (label, scores) =>
  val rejected = try {
    validateUserMismatchScores(scores)
    false
  } catch {
    case _: AssertionError => true
  }
  assert(rejected, s"Invalid score table should be rejected: $label")
}

unequalSegmentUPD.orderBy("segment").show(truncate = false)
println(f"Unequal-size overall UPD: $unequalOverall%.10f")
println(f"Pooled user average (comparison only): $pooledUserAverage%.10f")
println(s"All aggregation checks passed, including ${invalidScoreTables.size} rejected invalid inputs.")


+-------------------+---------+---+
|segment            |userCount|upd|
+-------------------+---------+---+
|blockbuster-focused|1        |0.0|
|diverse            |1        |0.0|
|niche-focused      |4        |1.0|
+-------------------+---------+---+



Unequal-size overall UPD: 0.3333333333


Pooled user average (comparison only): 0.6666666667


All aggregation checks passed, including 12 rejected invalid inputs.


unequalScores: DataFrame = [userId: int, segment: string ... 1 more field]
unequalSegmentUPD: DataFrame = [segment: string, userCount: bigint ... 1 more field]
unequalOverall: Double = 0.3333333333333333
pooledUserAverage: Double = 0.6666666666666666
invalidScoreTables: Seq[(String, DataFrame)] = List(
  ("duplicate user", [userId: int, segment: string ... 1 more field]),
  ("missing segment", [userId: int, segment: string ... 1 more field]),
  ("unknown segment", [userId: int, segment: string ... 1 more field]),
  ("missing user", [userId: int, segment: string ... 1 more field]),
  ("missing label", [userId: int, segment: string ... 1 more field]),
  ("missing score", [userId: int, segment: string ... 1 more field]),
  ("NaN score", [userId: int, segment: string ... 1 more field]),
  ("positive infinity", [userId: int, segment: string ... 1 more field]),
  ("negative infinity", [userId: int, segment: string ... 1 more field]),
  ("negative score", [userId: int, segment: string ... 1 m

## 7. Evaluate prepared input tables

`evaluateUPD` connects movie IDs to popularity labels, counts the H/M/T recommendation mix for each user, compares that mix with the user's historical profile, and returns `(userScores, segmentScores, overallScore)` as Spark DataFrames. It evaluates one method and one split at a time; it does not train ALS, assign popularity categories, construct historical profiles, or tune calibration.

Inputs must follow the schemas below. Extra columns, including ALS predictions and recommendation ranks, are ignored. Select the intended top-N list before calling the function; the default requires exactly 10 distinct movies per user.

| Input | Required columns |
|---|---|
| `profiles` | `userId: Int`, `head: Double`, `mid: Double`, `tail: Double` |
| `movieGroups` | `movieId: Int`, `popularityGroup: String` (`H`, `M`, or `T`) |
| `recommendations` | `userId: Int`, `movieId: Int` |
| `segments` | `userId: Int`, `segment: String` |
| `expectedUsers` | `userId: Int`, one row per intended evaluation user |

User segment labels are `blockbuster-focused`, `diverse`, and `niche-focused`. Profiles, segment assignments, and recommendation users must exactly match `expectedUsers`. Supply the cohort independently of the recommendation output so a missing recommendation list is detectable. For the baseline, the intended cohort is all users in the training split. Movie labels may cover a larger catalog, but each movie has exactly one valid label and every recommended movie must be covered.

All inputs must refer to the same IDs and training split. Movie popularity labels and historical rating-weighted proportions must use training data only. The evaluator validates structure and coverage; it cannot infer split provenance from these columns. For comparisons across methods, hold profiles, movie categories, segments, cohort, and list size fixed.

Spark column expressions implement the same zero-safe, base-2 Jensen–Shannon calculation as the scalar functions. This avoids serializing notebook state into Spark workers. Larger out-of-range results fail; rounding-sized excursions within `1e-9` are clipped to [0, 1]. Input profiles are not automatically renormalized. Calls validate inputs immediately; returned DataFrames remain Spark computations evaluated by actions.


In [10]:
import org.apache.spark.sql.Column
import org.apache.spark.sql.types.{DataType, IntegerType, DoubleType, StringType}
import org.apache.spark.sql.functions.{abs, greatest, least, log2, sum, when}

def requireColumns(
  table: DataFrame,
  label: String,
  fields: Seq[(String, DataType)]
): Unit = {
  fields.foreach { case (name, dataType) =>
    require(table.columns.count(_ == name) == 1,
      s"$label must contain exactly one column named $name.")
    require(table.schema(name).dataType == dataType,
      s"$label.$name must have type $dataType; convert it explicitly before evaluation.")
  }
}

def requireUniqueKeys(table: DataFrame, keys: Seq[String], label: String): Unit = {
  require(table.filter(keys.map(k => col(k).isNull).reduce(_ || _)).limit(1).count() == 0L,
    s"$label contains missing keys.")
  require(table.groupBy(keys.map(col): _*).count()
    .filter(col("count") > 1L).limit(1).count() == 0L,
    s"$label contains duplicate keys.")
}

def requireUserCoverage(table: DataFrame, expected: DataFrame, label: String): Unit = {
  val actual = table.select("userId").distinct()
  require(expected.join(actual, Seq("userId"), "left_anti").limit(1).count() == 0L,
    s"$label is missing expected users.")
  require(actual.join(expected, Seq("userId"), "left_anti").limit(1).count() == 0L,
    s"$label contains unexpected users; select the evaluation cohort explicitly.")
}

// Builds Spark expressions on the driver; no notebook closure is sent to workers.
def klContributionColumn(proportion: Column, reference: Column): Column = {
  when(proportion === 0.0, lit(0.0))
    .otherwise(proportion * log2(proportion / reference))
}

def evaluateUPD(
  profiles: DataFrame,
  movieGroups: DataFrame,
  recommendations: DataFrame,
  segments: DataFrame,
  expectedUsers: DataFrame,
  listSize: Int = 10
): (DataFrame, DataFrame, DataFrame) = {
  require(listSize > 0, "Recommendation list size must be positive.")
  val epsilon = 1e-9
  requireColumns(profiles, "profiles", Seq(
    "userId" -> IntegerType, "head" -> DoubleType,
    "mid" -> DoubleType, "tail" -> DoubleType))
  requireColumns(movieGroups, "movieGroups", Seq(
    "movieId" -> IntegerType, "popularityGroup" -> StringType))
  requireColumns(recommendations, "recommendations", Seq(
    "userId" -> IntegerType, "movieId" -> IntegerType))
  requireColumns(segments, "segments", Seq("userId" -> IntegerType, "segment" -> StringType))
  requireColumns(expectedUsers, "expectedUsers", Seq("userId" -> IntegerType))

  val cohort = expectedUsers.select("userId")
  val profileData = profiles.select("userId", "head", "mid", "tail")
  val groupData = movieGroups.select("movieId", "popularityGroup")
  val recommendationData = recommendations.select("userId", "movieId")
  val segmentData = segments.select("userId", "segment")

  requireUniqueKeys(cohort, Seq("userId"), "expectedUsers")
  require(cohort.limit(1).count() == 1L, "The evaluation cohort must not be empty.")
  requireUniqueKeys(profileData, Seq("userId"), "profiles")
  requireUniqueKeys(groupData, Seq("movieId"), "movieGroups")
  requireUniqueKeys(segmentData, Seq("userId"), "segments")
  requireUniqueKeys(recommendationData, Seq("userId", "movieId"), "recommendations")
  requireUserCoverage(profileData, cohort, "profiles")
  requireUserCoverage(segmentData, cohort, "segments")
  requireUserCoverage(recommendationData, cohort, "recommendations")

  val invalidProportion = Seq("head", "mid", "tail").map { name =>
    col(name).isNull || isnan(col(name)) || col(name) < 0.0 || col(name) > 1.0
  }.reduce(_ || _)
  require(profileData.filter(invalidProportion ||
    abs(col("head") + col("mid") + col("tail") - 1.0) > epsilon)
    .limit(1).count() == 0L, "Historical mixes must be finite, nonnegative, and sum to 1.")
  require(groupData.filter(col("popularityGroup").isNull ||
    !col("popularityGroup").isin("H", "M", "T")).limit(1).count() == 0L,
    "Movie popularity labels must be H, M, or T.")
  require(segmentData.filter(col("segment").isNull ||
    !col("segment").isin("blockbuster-focused", "diverse", "niche-focused"))
    .limit(1).count() == 0L, "User segment labels are missing or unknown.")
  require(segmentData.select("segment").distinct().count() == 3L,
    "All three user segments must be represented.")
  require(recommendationData.groupBy("userId").count()
    .filter(col("count") =!= listSize.toLong).limit(1).count() == 0L,
    s"Each user must have exactly $listSize distinct recommended movies.")
  require(recommendationData.select("movieId").distinct()
    .join(groupData.select("movieId"), Seq("movieId"), "left_anti")
    .limit(1).count() == 0L, "Some recommended movies have no popularity label.")

  val recommendationMixes = recommendationData.join(groupData, Seq("movieId"), "inner")
    .groupBy("userId")
    .agg(
      count("*").as("recommendationCount"),
      (sum(when(col("popularityGroup") === "H", 1.0).otherwise(0.0)) / count("*")).as("qHead"),
      (sum(when(col("popularityGroup") === "M", 1.0).otherwise(0.0)) / count("*")).as("qMid"),
      (sum(when(col("popularityGroup") === "T", 1.0).otherwise(0.0)) / count("*")).as("qTail")
    )
  val joined = profileData.join(recommendationMixes, Seq("userId"), "inner")
    .join(segmentData, Seq("userId"), "inner")
  val divergence = Seq("head" -> "qHead", "mid" -> "qMid", "tail" -> "qTail")
    .map { case (pName, qName) =>
      val p = col(pName)
      val q = col(qName)
      val m = (p + q) / 2.0
      (klContributionColumn(p, m) + klContributionColumn(q, m)) / 2.0
    }.reduce(_ + _)
  val rawScores = joined.withColumn("mismatch", divergence)
  require(rawScores.filter(col("mismatch").isNull || isnan(col("mismatch")) ||
    col("mismatch") < -epsilon || col("mismatch") > 1.0 + epsilon)
    .limit(1).count() == 0L, "Calculated mismatch is outside its numerical bounds.")
  // Clamp only rounding-sized excursions after rejecting larger violations.
  val userScores = rawScores.withColumn("mismatch",
    greatest(lit(0.0), least(lit(1.0), col("mismatch"))))
  requireUserCoverage(userScores, cohort, "computed scores")
  validateUserMismatchScores(userScores)
  val segmentScores = userScores.groupBy("segment")
    .agg(count("*").as("userCount"), avg("mismatch").as("upd"))
  val overallScore = segmentScores.agg(avg("upd").as("overallUPD"))
  (userScores, segmentScores, overallScore)
}


import org.apache.spark.sql.Column
import org.apache.spark.sql.types.{DataType, IntegerType, DoubleType, StringType}
import org.apache.spark.sql.functions.{abs, greatest, least, log2, sum, when}
defined function requireColumns
defined function requireUniqueKeys
defined function requireUserCoverage
defined function klContributionColumn
defined function evaluateUPD

## 8. Verify the complete flow with synthetic tables

This fixture starts with historical profiles, movie labels, recommendation movie IDs, segment assignments, and an independently specified evaluation cohort. All records are invented.

User 1 has an all-Head profile and all-Head recommendations (score 0). Users 2–3 reproduce the worked example (score approximately 0.218363597). Users 4–6 have all-Tail profiles but all-Head recommendations (score 1). Segment sizes are 1, 2, and 3; equal segment weighting gives overall UPD approximately 0.406121199. Checks verify the derived mixes, every score, segment means and counts, and overall score. Spark results are also compared with the scalar calculation.


In [11]:
val fixtureProfiles = Seq(
  (1, 1.0, 0.0, 0.0),
  (2, 0.2, 0.3, 0.5),
  (3, 0.2, 0.3, 0.5),
  (4, 0.0, 0.0, 1.0),
  (5, 0.0, 0.0, 1.0),
  (6, 0.0, 0.0, 1.0)
).toDF("userId", "head", "mid", "tail")
val fixtureSegments = Seq(
  (1, "blockbuster-focused"),
  (2, "diverse"), (3, "diverse"),
  (4, "niche-focused"), (5, "niche-focused"), (6, "niche-focused")
).toDF("userId", "segment")
val fixtureMovieGroups = (1 to 30).map { movieId =>
  (movieId, if (movieId <= 10) "H" else if (movieId <= 20) "M" else "T")
}.toDF("movieId", "popularityGroup")
val fixtureRecommendations = (1 to 6).flatMap { userId =>
  val movies = if (userId == 2 || userId == 3) Seq(1, 2, 3, 4, 5, 6, 7, 11, 12, 21)
               else (1 to 10)
  movies.map(movieId => (userId, movieId))
}.toDF("userId", "movieId")
val fixtureExpectedUsers = (1 to 6).toDF("userId")

val (fixtureUserUPD, fixtureSegmentUPD, fixtureOverallUPD) = evaluateUPD(
  fixtureProfiles, fixtureMovieGroups, fixtureRecommendations,
  fixtureSegments, fixtureExpectedUsers
)

val fixtureResults = fixtureUserUPD.orderBy("userId").collect()
assert(fixtureResults.length == 6)
fixtureResults.foreach { row =>
  val userId = row.getAs[Int]("userId")
  val isMixed = userId == 2 || userId == 3
  val expectedMix = if (isMixed) PopularityMix(0.7, 0.2, 0.1)
                    else PopularityMix(1.0, 0.0, 0.0)
  val expectedScore = if (userId == 1) 0.0 else if (isMixed) 0.2183635969629476 else 1.0
  assert(row.getAs[Long]("recommendationCount") == 10L)
  assert(math.abs(row.getAs[Double]("qHead") - expectedMix.head) <= tolerance)
  assert(math.abs(row.getAs[Double]("qMid") - expectedMix.mid) <= tolerance)
  assert(math.abs(row.getAs[Double]("qTail") - expectedMix.tail) <= tolerance)
  val scalarScore = popularityMismatch(
    PopularityMix(row.getAs[Double]("head"), row.getAs[Double]("mid"), row.getAs[Double]("tail")),
    expectedMix)
  assert(math.abs(row.getAs[Double]("mismatch") - expectedScore) <= tolerance)
  assert(math.abs(row.getAs[Double]("mismatch") - scalarScore) <= tolerance,
    "Spark and scalar calculations must agree.")
}
val fixtureExpectedSegments = Map(
  "blockbuster-focused" -> (1L, 0.0),
  "diverse" -> (2L, 0.2183635969629476),
  "niche-focused" -> (3L, 1.0))
fixtureSegmentUPD.collect().foreach { row =>
  val expected = fixtureExpectedSegments(row.getAs[String]("segment"))
  assert(row.getAs[Long]("userCount") == expected._1)
  assert(math.abs(row.getAs[Double]("upd") - expected._2) <= tolerance)
}
assert(math.abs(fixtureOverallUPD.first().getDouble(0) - 0.4061211989876492) <= tolerance)
fixtureUserUPD.select("userId", "segment", "qHead", "qMid", "qTail", "mismatch")
  .orderBy("userId").show(truncate = false)
fixtureSegmentUPD.orderBy("segment").show(truncate = false)
fixtureOverallUPD.show(truncate = false)
println("End-to-end synthetic UPD evaluation passed.")


+------+-------------------+-----+----+-----+------------------+
|userId|segment            |qHead|qMid|qTail|mismatch          |
+------+-------------------+-----+----+-----+------------------+
|1     |blockbuster-focused|1.0  |0.0 |0.0  |0.0               |
|2     |diverse            |0.7  |0.2 |0.1  |0.2183635969629475|
|3     |diverse            |0.7  |0.2 |0.1  |0.2183635969629475|
|4     |niche-focused      |1.0  |0.0 |0.0  |1.0               |
|5     |niche-focused      |1.0  |0.0 |0.0  |1.0               |
|6     |niche-focused      |1.0  |0.0 |0.0  |1.0               |
+------+-------------------+-----+----+-----+------------------+



+-------------------+---------+------------------+
|segment            |userCount|upd               |
+-------------------+---------+------------------+
|blockbuster-focused|1        |0.0               |
|diverse            |2        |0.2183635969629475|
|niche-focused      |3        |1.0               |
+-------------------+---------+------------------+



+------------------+
|overallUPD        |
+------------------+
|0.4061211989876492|
+------------------+



End-to-end synthetic UPD evaluation passed.


fixtureProfiles: DataFrame = [userId: int, head: double ... 2 more fields]
fixtureSegments: DataFrame = [userId: int, segment: string]
fixtureMovieGroups: DataFrame = [movieId: int, popularityGroup: string]
fixtureRecommendations: DataFrame = [userId: int, movieId: int]
fixtureExpectedUsers: DataFrame = [userId: int]
fixtureUserUPD: DataFrame = [userId: int, head: double ... 8 more fields]
fixtureSegmentUPD: DataFrame = [segment: string, userCount: bigint ... 1 more field]
fixtureOverallUPD: DataFrame = [overallUPD: double]
fixtureResults: Array[org.apache.spark.sql.Row] = Array(
  [1,1.0,0.0,0.0,10,1.0,0.0,0.0,blockbuster-focused,0.0],
  [2,0.2,0.3,0.5,10,0.7,0.2,0.1,diverse,0.2183635969629475],
  [3,0.2,0.3,0.5,10,0.7,0.2,0.1,diverse,0.2183635969629475],
  [4,0.0,0.0,1.0,10,1.0,0.0,0.0,niche-focused,1.0],
  [5,0.0,0.0,1.0,10,1.0,0.0,0.0,niche-focused,1.0],
  [6,0.0,0.0,1.0,10,1.0,0.0,0.0,niche-focused,1.0]
)
fixtureExpectedSegments: Map[String, (Long, Double)] = Map(
  "blockbuster-f

### Reject incomplete or ambiguous handoffs

These checks exercise failures that would otherwise drop users or multiply rows during joins, as well as missing category labels, incomplete lists, malformed profiles, and incompatible schemas. Inputs are rejected rather than silently filtered or repaired.


In [12]:
// Each case changes one input while preserving the other valid fixture tables.
val invalidIntegrationCases = Seq(
  ("missing movie label", fixtureProfiles, fixtureMovieGroups.filter(col("movieId") =!= 1), fixtureRecommendations, fixtureSegments, fixtureExpectedUsers),
  ("duplicate movie mapping", fixtureProfiles, fixtureMovieGroups.union(fixtureMovieGroups.filter(col("movieId") === 1)), fixtureRecommendations, fixtureSegments, fixtureExpectedUsers),
  ("unknown movie label", fixtureProfiles, fixtureMovieGroups.withColumn("popularityGroup", lit("unknown")), fixtureRecommendations, fixtureSegments, fixtureExpectedUsers),
  ("missing profile", fixtureProfiles.filter(col("userId") =!= 1), fixtureMovieGroups, fixtureRecommendations, fixtureSegments, fixtureExpectedUsers),
  ("duplicate profile", fixtureProfiles.union(fixtureProfiles.filter(col("userId") === 1)), fixtureMovieGroups, fixtureRecommendations, fixtureSegments, fixtureExpectedUsers),
  ("invalid profile sum", fixtureProfiles.withColumn("head", lit(0.9)), fixtureMovieGroups, fixtureRecommendations, fixtureSegments, fixtureExpectedUsers),
  ("missing user recommendations", fixtureProfiles, fixtureMovieGroups, fixtureRecommendations.filter(col("userId") =!= 6), fixtureSegments, fixtureExpectedUsers),
  ("short recommendation list", fixtureProfiles, fixtureMovieGroups, fixtureRecommendations.filter(!(col("userId") === 1 && col("movieId") === 1)), fixtureSegments, fixtureExpectedUsers),
  ("duplicate recommendation", fixtureProfiles, fixtureMovieGroups, fixtureRecommendations.union(fixtureRecommendations.limit(1)), fixtureSegments, fixtureExpectedUsers),
  ("unexpected user", fixtureProfiles, fixtureMovieGroups, fixtureRecommendations.union(Seq((99, 1)).toDF("userId", "movieId")), fixtureSegments, fixtureExpectedUsers),
  ("missing segment assignment", fixtureProfiles, fixtureMovieGroups, fixtureRecommendations, fixtureSegments.filter(col("userId") =!= 1), fixtureExpectedUsers),
  ("duplicate segment assignment", fixtureProfiles, fixtureMovieGroups, fixtureRecommendations, fixtureSegments.union(fixtureSegments.limit(1)), fixtureExpectedUsers),
  ("unknown segment label", fixtureProfiles, fixtureMovieGroups, fixtureRecommendations, fixtureSegments.withColumn("segment", lit("unknown")), fixtureExpectedUsers),
  ("wrong identifier type", fixtureProfiles.withColumn("userId", col("userId").cast("string")), fixtureMovieGroups, fixtureRecommendations, fixtureSegments, fixtureExpectedUsers),
  ("empty cohort", fixtureProfiles, fixtureMovieGroups, fixtureRecommendations, fixtureSegments, fixtureExpectedUsers.limit(0))
)
invalidIntegrationCases.foreach { case (label, profiles, groups, recommendations, segments, users) =>
  val rejected = try {
    evaluateUPD(profiles, groups, recommendations, segments, users)
    false
  } catch {
    case _: IllegalArgumentException => true
  }
  assert(rejected, s"Invalid integration input should be rejected: $label")
}
println(s"All ${invalidIntegrationCases.size} integration rejection checks passed.")


All 15 integration rejection checks passed.


invalidIntegrationCases: Seq[(String, DataFrame, DataFrame, DataFrame, DataFrame, org.apache.spark.sql.Dataset[org.apache.spark.sql.Row])] = List(
  (
    "missing movie label",
    [userId: int, head: double ... 2 more fields],
    [movieId: int, popularityGroup: string],
    [userId: int, movieId: int],
    [userId: int, segment: string],
    [userId: int]
  ),
  (
    "duplicate movie mapping",
    [userId: int, head: double ... 2 more fields],
    [movieId: int, popularityGroup: string],
    [userId: int, movieId: int],
    [userId: int, segment: string],
    [userId: int]
  ),
  (
    "unknown movie label",
    [userId: int, head: double ... 2 more fields],
    [movieId: int, popularityGroup: string],
    [userId: int, movieId: int],
    [userId: int, segment: string],
    [userId: int]
  ),
  (
    "missing profile",
    [userId: int, head: double ... 2 more fields],
    [movieId: int, popularityGroup: string],
    [userId: int, movieId: int],
    [userId: int, segment: string],


## 9. Use real inputs

The synthetic run is self-contained. Real evaluation additionally requires prepared historical profiles, movie popularity labels, segment assignments, and an expected-user cohort from the same split as the recommendations. Those input files are not bundled with this notebook. Column names and types can be adapted explicitly to the documented schema without changing the metric.

After running the definition cells above, load the full baseline `top-10-baseline/` Parquet directory and the prepared input tables, then call the evaluator before shutting down Spark. Example paths below are placeholders and are not executed automatically:

```scala
val profiles = spark.read.parquet("/path/to/run/user-popularity-profiles")
val movieGroups = spark.read.parquet("/path/to/run/movie-popularity-groups")
val segments = spark.read.parquet("/path/to/run/user-segments")
val expectedUsers = spark.read.parquet("/path/to/run/evaluation-users")
val recommendations = spark.read.parquet("/path/to/baseline-run/top-10-baseline")

val (userScores, segmentScores, overallScore) = evaluateUPD(
  profiles, movieGroups, recommendations, segments, expectedUsers
)
segmentScores.orderBy("segment").show(truncate = false)
overallScore.show(truncate = false)
```

`userScores` retains historical proportions (`head`, `mid`, `tail`), recommendation proportions (`qHead`, `qMid`, `qTail`), `recommendationCount`, `userId`, `segment`, and `mismatch`. Segment results contain `segment`, `userCount`, and `upd`; the overall result contains one `overallUPD` value. Keep method and split/run identity with any saved results. Generated outputs belong under the ignored `outputs/` directory.

The baseline top-100 candidate pool is not a top-10 list; it is used by calibration. For each calibrated method, supply its final top-10 user–movie pairs in place of baseline recommendations. Test ratings and ALS predicted relevance scores are not needed by this metric. The implementation measures popularity alignment; relevance evaluation is separate.


## 10. Stop the Spark session

Run this cell when Spark work is complete to release the session's resources. Restart the kernel and run the notebook in order for a fresh environment check.


In [13]:
spark.stop()
println("Spark session stopped.")


Spark session stopped.
